# Assessment Notebook

## Problem 1: Representing SHA-256 Data

This problem involves: research implementing SHA-256, investigate how the input, output, and intermediate data can be represented in Python.

### 32-bit words

SHA-256 operates on groups of 32 bits called words. Each word is an unsigned integer with a range from (0) to (2^{32} - 1). A 32-bit word can be represented using eight hexadecimal digits, as each hexadecimal digit represents four bits.

For example:

In [1]:
import numpy as np

word = np.uint32(0x12345678)

print(word)
print(word.dtype)

305419896
uint32


`numpy.uint32` is appropriate because it explicitly represents an unsigned 32-bit integer, matching the word size used by SHA-256. This is preferable to a floating-point type because SHA-256 relies on integer and bitwise operations such as XOR, AND and bit shifting.

The fixed 32-bit representation is also important because SHA-256 uses arithmetic modulo (2^{32}). For example:

In [2]:
import numpy as np

word = np.uint32(0xFFFFFFFF)
one = np.uint32(1)

result = np.uint32((int(word) + int(one)) & 0xFFFFFFFF)

print(f"{result:08x}")

00000000


This results in `00000000`, demonstrating the 32-bit wraparound from (2^{32} - 1 + 1).

Hexadecimal is mainly used as a convenient way of displaying the word; the underlying value is still a 32-bit unsigned integer.

### Sequences of 32-bit words

A SHA-256 word is exactly 32 bits, so it can represent an unsigned integer in the range:

$$
0 \leq x \leq 2^{32} - 1
$$

In Python, a sequence of these words can be represented using a NumPy array with the `numpy.uint32` data type. This ensures that each element occupies exactly 32 bits and that arithmetic and bitwise operations behave similarly to the operations required by SHA-256.

For example:

In [3]:
import numpy as np

words = np.array([
    0x12345678,
    0xABCDEF01,
    0x00000000,
    0xFFFFFFFF
], dtype=np.uint32)

print(words)
print(words.dtype)

[ 305419896 2882400001          0 4294967295]
uint32


This produces an array where every element is explicitly represented as a 32-bit unsigned integer.

The hexadecimal notation is useful because eight hexadecimal digits correspond to 32 bits:
$$
0x12345678=00010010 00110100 01010110 01111000
$$
A NumPy `uint32` is particularly suitable for SHA-256 because the algorithm requires operations such as addition modulo $2^{32}$, bitwise AND, OR, XOR, and rotations.

### Input Messages

SHA-256 accepts an input message of arbitrary length. The message is ultimately processed as a sequence of bytes, so bytes is an appropriate representation in Python.

A text message can initially be represented as a Python `str`, but it must be converted into bytes before being passed to SHA-256. This can be done using the `.encode()` method. UTF-8 encoding is suitable for representing text, with each character being converted into one or more bytes.

For example, the message `"abc"` can be represented as follows:

In [4]:
# Example input message 
message = "abc" 

# Convert the message into UTF-8 bytes 
message_bytes = message.encode("utf-8") 

print("Message:", message) 
print("Bytes:", message_bytes) 
print("Length:", len(message_bytes))

Message: abc
Bytes: b'abc'
Length: 3


The resulting `bytes` object contains the three byte values corresponding to the characters `a`, `b`, and `c`. These bytes form the input data that can then be processed by the SHA-256 algorithm.

The message `"abc"` is also used as an example in the SHA-256 specification ([FIPS 180-4, Section 5.1.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=18)).

### 512-bit message blocks

SHA-256 processes messages in blocks of 512 bits, equivalent to 64 bytes. According to ([NIST FIPS 180-4, Section 5.2.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=17)), the padded message is divided into 512-bit blocks, with each block consisting of sixteen 32-bit words.

In Python, a block can be represented using a `bytes` object containing exactly 64 bytes. This is appropriate because each byte represents eight bits, making it straightforward to store and manipulate the binary input. The bytes can then be converted into a NumPy array of sixteen `uint32` values, matching the word structure used by SHA-256.

For example:

In [9]:
import numpy as np

# Create an example 512-bit message block
block = bytes(range(64))

print("Block size in bytes:", len(block))
print("Block size in bits:", len(block) * 8)
print("First 8 bytes:", block[:8])

Block size in bytes: 64
Block size in bits: 512
First 8 bytes: b'\x00\x01\x02\x03\x04\x05\x06\x07'


The block contains 64 bytes, giving a total of 512 bits. The `bytes` representation is useful for storing the original message data, while NumPy can be used to represent the sixteen 32-bit words required for processing.

Each group of four bytes must be interpreted in big-endian order, meaning that the most significant byte comes first. This follows the convention specified in FIPS 180-4.

In [10]:
# Convert each group of four bytes into a 32-bit word
words = np.array([
    int.from_bytes(block[i:i + 4], byteorder="big")
    for i in range(0, 64, 4)
], dtype=np.uint32)

print("Number of words:", len(words))
print("Data type:", words.dtype)
print("First four words:", [f"{word:08x}" for word in words[:4]])

Number of words: 16
Data type: uint32
First four words: ['00010203', '04050607', '08090a0b', '0c0d0e0f']


This produces sixteen unsigned 32-bit integers. For this example, the first four words are 00010203, 04050607, 08090a0b, and 0c0d0e0f.

Using `numpy.uint32` is appropriate because SHA-256 performs its operations on 32-bit words, including bitwise operations and addition modulo $(2^{32})$. The array representation therefore provides a convenient way to organise the message block for subsequent stages of the algorithm.

The original bytes object and the array of words serve different purposes: bytes preserves the input as a sequence of eight-bit values, while the NumPy array provides the sixteen 32-bit words that the SHA-256 compression function uses.

### The final 256-bit hash value

SHA-256 produces a fixed-length message digest of 256 bits, regardless of the length of the input message. The digest is the final output of the hashing algorithm and provides a compact representation of the input data. NIST specifies the digest size and algorithm in the Secure Hash Standard ([FIPS 180-4, Section 6.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=27)).

Since one byte contains eight bits, the digest occupies:

$$
\frac{256\text{ bits}}{8\text{ bits per byte}} = 32\text{ bytes}
$$

A SHA-256 digest is commonly displayed in hexadecimal form. Each hexadecimal digit represents four bits, so the complete digest contains:

$$
\frac{256\text{ bits}}{4\text{ bits per hexadecimal digit}} = 64\text{ hexadecimal digits}
$$

In Python, the digest can be represented as a bytes object containing 32 bytes and converted into a hexadecimal string using the .hex() method. This is appropriate because bytes stores the digest as a sequence of eight-bit values, while hexadecimal provides a readable representation of the same binary data. Each byte requires two hexadecimal digits, so the hexadecimal representation is twice as long as the byte sequence.

For example, the SHA-256 test message "abc" produces the following digest. This value is defined by the SHA-256 specification and is a 256-bit output, as described in ([NIST FIPS 180-4, Section 6.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=27)).

In [1]:
# Example final SHA-256 hash value for "abc"
hash_value = bytes.fromhex(
    "ba7816bf8f01cfea414140de5dae2223"
    "b00361a396177a9cb410ff61f20015ad"
)

print("Hash:", hash_value.hex())
print("Size in bytes:", len(hash_value))
print("Size in bits:", len(hash_value) * 8)
print("Hexadecimal digits:", len(hash_value.hex()))

Hash: ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad
Size in bytes: 32
Size in bits: 256
Hexadecimal digits: 64


The code uses `bytes.fromhex()` to convert the hexadecimal representation into its corresponding sequence of bytes. It does not calculate the digest; instead, it stores a known test value so that its representation and size can be examined without using a hashing library.

Internally, SHA-256 maintains its hash value as eight 32-bit words. After all message blocks have been processed, these words are concatenated in order to form the final 256-bit digest. The total size is therefore:

$$
8 \times 32 = 256\text{ bits}
$$

This connects the final digest representation to the intermediate data structures used by the algorithm.

Therefore, a `bytes` object is suitable for representing the complete digest, a hexadecimal string is useful for displaying it, and eight 32-bit words describe its structure within the SHA-256 algorithm.

### Big-endian integers

Big-endian is a byte-order convention in which the most significant byte is stored or interpreted first. In other words, the byte containing the highest-order bits appears before the bytes containing the lower-order bits.

SHA-256 uses big-endian byte ordering when interpreting message data as 32-bit words. According to [NIST FIPS 180-4, Section 5.2.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=17), each 512-bit message block is divided into sixteen 32-bit words. Each word is formed by interpreting its four bytes in big-endian order.

For example, consider the following four bytes:
$$
12, 34, 56, 78
$$

When interpreted as a big-endian integer, the first byte (0x12) is the most significant byte, producing the integer 0x12345678. The corresponding calculation is:

$$
(0x12 \times 2^{24}) +
(0x34 \times 2^{16}) +
(0x56 \times 2^8) +
0x78
$$

In Python, the `int.from_bytes()` method can convert a sequence of bytes into an integer using a specified byte order. This is appropriate for SHA-256 because it allows message bytes to be interpreted according to the algorithm's required big-endian convention.

For example:

In [2]:
# Four bytes representing a 32-bit word
data = bytes.fromhex("12345678")

# Interpret the bytes as a big-endian integer
big_endian_value = int.from_bytes(data, byteorder="big")

# Interpret the same bytes as a little-endian integer
little_endian_value = int.from_bytes(data, byteorder="little")

print("Big-endian:", f"{big_endian_value:08x}")
print("Little-endian:", f"{little_endian_value:08x}")

Big-endian: 12345678
Little-endian: 78563412


The two results differ because the same sequence of bytes is interpreted in a different order. In big-endian representation, the first byte is the most significant; in little-endian representation, the first byte is the least significant.

Using `int.from_bytes(data, byteorder="big")` is therefore suitable when converting message data into the 32-bit words required by SHA-256. Explicitly specifying the byte order also makes the code easier to understand and avoids relying on the computer's native byte order.

## Problem 2: SHA-256 Bitwise Operations

This problem involves: implementing the Boolean and bitwise functions used by SHA-256

### 1. `Ch(x, y, z)`: Choose Function

The SHA-256 Choose function operates on three 32-bit words, x, y, and z.

It is defined as:

$$
\operatorname{Ch}(x,y,z) = (x \land y) \oplus (\neg x \land z)
$$

Where:

$(\land)$ represents bitwise AND.

$(\oplus)$ represents bitwise XOR.

$(\neg)$ represents bitwise NOT.

For each bit position, the function selects the corresponding bit from y if the bit in x is 1; otherwise, it selects the bit from z.

In SHA-256, these inputs are 32-bit words. NumPy's `uint32` type ensures that the values are represented as 32-bit unsigned integers.

Reference: [NIST FIPS 180-4, Section 4.1.2](https://doi.org/10.6028/NIST.FIPS.180-4), which defines the SHA-256 logical functions.

#### IMPLEMENTATION OF SHA-256 CHOOSE FUNCTION

In [ ]:
import numpy as np


def Ch(x: np.uint32, y: np.uint32, z: np.uint32) -> np.uint32:
    """Return the SHA-256 Choose function for three 32-bit words."""
    x = np.uint32(x)
    y = np.uint32(y)
    z = np.uint32(z)

    mask = np.uint32(0xFFFFFFFF)
    return np.uint32(
        ((x & y) ^ ((~x & mask) & z))
    )

Python and NumPy bitwise NOT can produce different-looking results depending on the integer type. The mask `0xFFFFFFFF` ensures that the complement of x is restricted to 32 bits.

#### Testing the Choose Function

The following tests check the function using values that make its expected behaviour easy to verify:

1. If `x` contains only zeros, the result should equal `z`.
2. If `x` contains only ones, the result should equal `y`.
3. If `y` and `z` are identical, the result should equal their shared value, regardless of `x`.
4. The result must be a NumPy 32-bit unsigned integer.

#### Unit Tests

In [3]:

# Test 1: All bits of x are zero, so choose z.
assert Ch(
    np.uint32(0x00000000),
    np.uint32(0x12345678),
    np.uint32(0xABCDEF01),
) == np.uint32(0xABCDEF01)

# Test 2: All bits of x are one, so choose y.
assert Ch(
    np.uint32(0xFFFFFFFF),
    np.uint32(0x12345678),
    np.uint32(0xABCDEF01),
) == np.uint32(0x12345678)

# Test 3: Identical y and z must produce that value.
assert Ch(
    np.uint32(0xA5A5A5A5),
    np.uint32(0x12345678),
    np.uint32(0x12345678),
) == np.uint32(0x12345678)

# Test 4: Verify the result type.
result = Ch(
    np.uint32(0xAAAAAAAA),
    np.uint32(0xCCCCCCCC),
    np.uint32(0x55555555),
)
assert isinstance(result, np.uint32)

print("All Ch function tests passed.")


All Ch function tests passed.


### `Maj(x, y, z)`: Majority Function

The SHA-256 Majority function operates on three 32-bit words, x, y, and z.

It is defined as:

$$
\operatorname{Maj}(x,y,z) = (x \land y) \oplus (x \land z) \oplus (y \land z)
$$

Where:

$(\land)$ represents bitwise AND.

$(\oplus)$ represents bitwise XOR.

For each bit position, the function returns 1 if at least two of the three input bits are 1; otherwise, it returns 0.

NumPy's `uint32` type ensures that the values are represented as 32-bit unsigned integers.

Reference: [NIST FIPS 180-4, Section 4.1.2](https://doi.org/10.6028/NIST.FIPS.180-4), which defines the SHA-256 logical functions.

#### IMPLEMENTATION OF SHA-256 MAJORITY FUNCTION

In [ ]:
import numpy as np


def Maj(x: np.uint32, y: np.uint32, z: np.uint32) -> np.uint32:
    """Return the SHA-256 Majority function for three 32-bit words."""
    x = np.uint32(x)
    y = np.uint32(y)
    z = np.uint32(z)

    return np.uint32((x & y) ^ (x & z) ^ (y & z))


#### Testing the Majority Function
The following tests verify the expected behaviour:

1. If all inputs are zero, the result should be zero.

2. If all inputs are one, the result should be one.

3. If two inputs are identical and the third is their bitwise complement, the result should equal the repeated input.

4. The result must be a NumPy 32-bit unsigned integer.

#### Unit Tests

In [6]:

# Test 1: All inputs are zero.
assert Maj(
    np.uint32(0x00000000),
    np.uint32(0x00000000),
    np.uint32(0x00000000),
) == np.uint32(0x00000000)

# Test 2: All inputs are one.
assert Maj(
    np.uint32(0xFFFFFFFF),
    np.uint32(0xFFFFFFFF),
    np.uint32(0xFFFFFFFF),
) == np.uint32(0xFFFFFFFF)

# Test 3: Two inputs match, so their value is selected.
assert Maj(
    np.uint32(0xAAAAAAAA),
    np.uint32(0xAAAAAAAA),
    np.uint32(0x55555555),
) == np.uint32(0xAAAAAAAA)

# Test 4: Verify the result type.
result = Maj(
    np.uint32(0xAAAAAAAA),
    np.uint32(0xCCCCCCCC),
    np.uint32(0xF0F0F0F0),
)
assert isinstance(result, np.uint32)

print("All Maj function tests passed.")


All Maj function tests passed.


### `Sigma0(x)`: Uppercase Sigma Zero Function

The SHA-256 function $(\Sigma_0(x))$ operates on a 32-bit word and is defined as:

$$
\Sigma_0(x) = \operatorname{ROTR}^{2}(x)
\oplus \operatorname{ROTR}^{13}(x)
\oplus \operatorname{ROTR}^{22}(x)
$$

Where:

$(\operatorname{ROTR}^{n}(x))$ rotates the bits of x to the right by n positions.

$(\oplus)$ represents bitwise XOR.

Rotation moves the bits that leave the right-hand side back to the left-hand side, unlike a normal right shift.

Reference: [NIST FIPS 180-4, Section 4.1.2](https://doi.org/10.6028/NIST.FIPS.180-4).

In [8]:

import numpy as np


def _rotr(x: np.uint32, n: int) -> np.uint32:
    """Rotate a 32-bit unsigned integer right by n bits."""
    x = np.uint32(x)
    return np.uint32((x >> n) | (x << (32 - n)))


def Sigma0(x: np.uint32) -> np.uint32:
    """Return the SHA-256 uppercase Sigma zero function."""
    x = np.uint32(x)
    return np.uint32(
        _rotr(x, 2) ^ _rotr(x, 13) ^ _rotr(x, 22)
    )


#### Testing the Sigma0 Function

The following tests verify that the function behaves correctly:

1. An input of zero should produce zero.

2. An input of all ones should produce all ones.

3. An input of 0x00000001 should produce the expected result after the specified rotations and XOR operations.

4. The output should be a NumPy 32-bit unsigned integer (np.uint32).

#### Unit Tests

In [9]:

# Test 1: Zero input.
assert Sigma0(np.uint32(0x00000000)) == np.uint32(0x00000000)

# Test 2: All bits set.
assert Sigma0(np.uint32(0xFFFFFFFF)) == np.uint32(0xFFFFFFFF)

# Test 3: Verify rotations for an input of 1.
assert Sigma0(np.uint32(0x00000001)) == np.uint32(0x40080400)

# Test 4: Verify the result type.
assert isinstance(Sigma0(np.uint32(0x12345678)), np.uint32)

print("All Sigma0 function tests passed.")


All Sigma0 function tests passed.


### `sigma0(x)`: Lowercase Sigma Zero Function

The SHA-256 function $\sigma_0(x)$ operates on a 32-bit word and is defined as:

$$
\sigma_0(x) = \operatorname{ROTR}^{7}(x)
\oplus \operatorname{ROTR}^{18}(x)
\oplus \operatorname{SHR}^{3}(x)
$$

Where:

$(\operatorname{ROTR}^{n}(x))$ rotates the bits of x to the right by n positions.

$(\operatorname{SHR}^{n}(x))$ shifts the bits of x to the right by n positions, filling the empty positions with zeros.

$(\oplus)$ represents bitwise XOR.

Unlike a right rotation, a right shift discards the bits that leave the right-hand side. The function combines two rotations and one shift using XOR.

NumPy's `uint32` type ensures that the values are represented as 32-bit unsigned integers.

Reference: [NIST FIPS 180-4, Section 4.1.2](https://doi.org/10.6028/NIST.FIPS.180-4).

#### IMPLEMENTATION OF SHA-256 LOWERCASE SIGMA ZERO FUNCTION

In [ ]:
def sigma0(x: np.uint32) -> np.uint32:
    """Return the SHA-256 lowercase Sigma zero function."""
    x = np.uint32(x)

    return np.uint32(
        _rotr(x, 7) ^ _rotr(x, 18) ^ (x >> 3)
    )

#### Testing the sigma0 Function

The following tests verify that the function behaves correctly:

1. An input of zero should produce zero.

2. An input of all ones should produce the expected result after the rotations, shift and XOR operations.

3. An input of 0x00000001 should produce the expected result after the specified rotations, shift and XOR operations.

4. The output should be a NumPy 32-bit unsigned integer (np.uint32).

#### Unit Tests

In [ ]:
# Test 1: Zero input.
assert sigma0(np.uint32(0x00000000)) == np.uint32(0x00000000)

# Test 2: All bits set.
assert sigma0(np.uint32(0xFFFFFFFF)) == np.uint32(0x1FFFFFFF)

# Test 3: Verify rotations and shift for an input of 1.
assert sigma0(np.uint32(0x00000001)) == np.uint32(0x02004000)

# Test 4: Verify the result type.
assert isinstance(sigma0(np.uint32(0x12345678)), np.uint32)

print("All sigma0 function tests passed.")

## Problem 3: Generating the SHA-256 Constants

This problem involves: 

## Problem 4: Padding and Parsing Messages

This problem involves: 

## Problem 5: The SHA-256 Compression Function

This problem involves: 

## Problem 6: Complete SHA-256

This problem involves: 

## End